[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science-full/blob/main/notebooks/Module_02_Data_Acquisition/M2_01_amsterdam_bike_api.ipynb)

# 🚴 Module 02: Amsterdam Bike Data Acquisition

**Purpose**: Learn to fetch real-time bike availability data from the CityBikes API  
**Module**: Module 02 - Data Acquisition  
**Author**: [Your Name]  
**Date**: 2026-01-15

---

## 📋 Overview

In this notebook, you will:
- [ ] Understand the CityBikes API structure
- [ ] Fetch real-time bike availability data for Amsterdam
- [ ] Parse and explore JSON responses
- [ ] Implement error handling for API requests
- [ ] Save raw data with proper documentation
- [ ] Validate data quality

**Goal**: Build confidence fetching data from REST APIs!

**Estimated Time**: 90-120 minutes

---

## 🎓 Learning Path Note

**This Module**: Focus on exploring and understanding data acquisition concepts using notebook-based experiments. Write code directly in cells to learn.

**Module 8 - Automation**: You'll learn to convert these exploration patterns into production-ready scripts (`src/data_acquisition.py`). For now, the code in `src/` serves as reference material showing professional patterns.

**Learning Strategy**: 
- ✅ DO: Experiment and learn in this notebook
- ⏸️ LATER: Productionize code into reusable scripts (Module 8)

---

---

## 📖 Part 1: Understanding the API

### What is CityBikes API?

[CityBikes](http://api.citybik.es/v2/) is a **free, open API** that aggregates bike-sharing data from cities worldwide.

**Key Features**:
- ✅ **No authentication required** - just make HTTP requests
- ✅ **Real-time data** - updated every few minutes
- ✅ **Global coverage** - 600+ cities
- ✅ **JSON format** - easy to parse with Python

### API Endpoints

**Base URL**: `http://api.citybik.es/v2/`

| Endpoint | Description | Example |
|----------|-------------|---------|
| `/networks` | List all available networks | `GET /v2/networks` |
| `/networks/{id}` | Get specific network details | `GET /v2/networks/ns-bikes` |

### Amsterdam Network

Amsterdam has multiple bike-sharing systems. For this project, we'll use:
- **Network ID**: `ns-bikes` (NS Bike sharing at train stations)
- **Alternative**: `donkey-republic-amsterdam` (Donkey Republic bikes)

### Data Structure

The API returns JSON with this structure:
```json
{
  "network": {
    "id": "ns-bikes",
    "name": "NS Bike",
    "location": {
      "city": "Amsterdam",
      "country": "NL",
      "latitude": 52.3676,
      "longitude": 4.9041
    },
    "stations": [
      {
        "id": "station-001",
        "name": "Amsterdam Centraal",
        "latitude": 52.3791,
        "longitude": 4.9003,
        "free_bikes": 5,
        "empty_slots": 15,
        "timestamp": "2026-01-15T10:30:00.000000Z"
      }
    ]
  }
}
```

**Key Fields**:
- `free_bikes`: Number of bikes available to rent
- `empty_slots`: Number of empty docking slots
- `timestamp`: When the data was collected

---

## 🔧 Part 2: Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import os
import sys
import json
from datetime import datetime
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
    # Uncomment to clone repository
    # !git clone https://github.com/vinculum3141-ship-it/bike-availability-data-science-full.git
    # %cd bike-availability-data-science-full
    # !pip install -q -r requirements.txt
else:
    print("📍 Running locally")

# Add project root to path
project_root = os.path.abspath('../..' if 'notebooks' in os.getcwd() else '.')
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Part 3: Configuration

Define API endpoints and file paths.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ═══════════════════════════════════════════════════════════

# API Configuration
BASE_URL = "http://api.citybik.es/v2"
NETWORK_ID = "ns-bikes"  # NS Bike sharing at Amsterdam train stations
NETWORK_URL = f"{BASE_URL}/networks/{NETWORK_ID}"

# Alternative network (uncomment to try different bike system)
# NETWORK_ID = "donkey-republic-amsterdam"

# File paths
DATA_DIR = Path('../../data/raw') if 'notebooks' in os.getcwd() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output filename with timestamp
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUTPUT_FILE = DATA_DIR / f'amsterdam_bike_{TIMESTAMP}.json'

print("✅ Configuration set")
print(f"🌐 API URL: {NETWORK_URL}")
print(f"💾 Output file: {OUTPUT_FILE}")

---

## 🌐 Part 4: Fetch Data from API

### 🧠 Your Task: Make Your First API Request

**What You've Learned in Module 1:**
- In [M1_R1_fetching_bike_data.ipynb](../Module_01_Introduction/M1_R1_fetching_bike_data.ipynb) and [M1_03_open_data_sources.ipynb](../Module_01_Introduction/M1_03_open_data_sources.ipynb), you saw how to:
  - Make API requests with `requests.get()`
  - Check status codes (`response.status_code == 200`)
  - Parse JSON with `response.json()`
  - Handle basic errors

**Now It's Your Turn:**
Complete the code below to fetch bike data from the CityBikes API. Use what you learned in Module 1!

**Steps:**
1. Make a GET request to `NETWORK_URL`
2. Check if the status code is 200
3. Parse the JSON response
4. Print basic information about the response

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. FETCH DATA - YOUR TASK
# ═══════════════════════════════════════════════════════════

print("📡 Fetching bike data from CityBikes API...")
print(f"🔗 URL: {NETWORK_URL}")

# TODO: Make the API request
# Hint: Use requests.get() like you did in Module 1
response = None  # Replace with your code

# TODO: Check if request was successful
# Hint: Check response.status_code == 200
print(f"\n📊 Response Status: ???")  # Replace with actual status

# TODO: If successful, parse JSON and print info
# Hint: Use response.json() to parse
# Hint: Print network name, location, and number of stations

# Example structure (you saw this in M1_R1):
# data = response.json()
# print(f"Network: {data['network']['name']}")
# print(f"Number of stations: {len(data['network']['stations'])}")

# Your code here...

### Step 2: Explore the JSON Structure

Let's look at the first station to understand the data structure.

In [ ]:
# ═══════════════════════════════════════════════════════════
# SOLUTION: FETCH DATA
# ═══════════════════════════════════════════════════════════

print("📡 Fetching bike data from CityBikes API...")
print(f"🔗 URL: {NETWORK_URL}")

# Make the API request
response = requests.get(NETWORK_URL)

# Check if request was successful
print(f"\n📊 Response Status: {response.status_code}")

if response.status_code == 200:
    print("✅ Success! Data fetched successfully")
    
    # Parse JSON response
    data = response.json()
    
    print(f"\n📦 Response keys: {list(data.keys())}")
    print(f"🌍 Network: {data['network']['name']}")
    print(f"📍 Location: {data['network']['location']['city']}, {data['network']['location']['country']}")
    print(f"🚴 Number of stations: {len(data['network']['stations'])}")
else:
    print(f"❌ Error: HTTP {response.status_code}")
    print(f"Message: {response.text}")

### ✅ Solution: Fetch Data

Run this cell after attempting the task above to see the complete solution.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. EXPLORE JSON STRUCTURE
# ═══════════════════════════════════════════════════════════

# Get first station as example
first_station = data['network']['stations'][0]

print("🔍 First Station Data:")
print("=" * 60)
print(json.dumps(first_station, indent=2))

print("\n" + "=" * 60)
print("📊 Station Fields:")
for key, value in first_station.items():
    print(f"  • {key:20s}: {type(value).__name__:10s} = {value}")

### 🧠 Learner Task 1: Implement Error Handling

**Your Task**: Improve the API request with proper error handling.

Complete the function below to:
1. Add a timeout parameter
2. Handle connection errors
3. Handle HTTP errors
4. Return None if anything fails

**Hint**: Use `try/except` blocks with `requests.exceptions.RequestException`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. LEARNER TASK 1: ERROR HANDLING
# ═══════════════════════════════════════════════════════════

def fetch_bike_data_safe(network_id, timeout=10):
    """
    Fetch bike data with error handling.
    
    Parameters:
    -----------
    network_id : str
        The network ID (e.g., 'ns-bikes')
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    url = f"{BASE_URL}/networks/{network_id}"
    
    # TODO: Implement error handling
    # Hint: Use try/except with requests.exceptions.RequestException
    # Hint: Use response.raise_for_status() to check HTTP errors
    
    try:
        # TODO: Make request with timeout
        response = None  # Replace with actual request
        
        # TODO: Raise exception for bad HTTP status (4xx, 5xx)
        
        # TODO: Parse and return JSON
        return None  # Replace with actual return
        
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.ConnectionError:
        print("🔌 Connection Error: Could not connect to API")
        return None
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error: {e}")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Request Error: {e}")
        return None
        
    except json.JSONDecodeError:
        print("📝 JSON Error: Could not parse response")
        return None


# Test your function
print("Testing error handling...")
test_data = fetch_bike_data_safe(NETWORK_ID, timeout=10)

if test_data:
    print(f"✅ Success! Fetched {len(test_data['network']['stations'])} stations")
else:
    print("❌ Function needs implementation")

### ✅ Solution Check: Error Handling

Run this cell to see a working solution (after you've tried yourself!).

### Pattern 2: Rate Limiting for Bulk Requests

When fetching data from multiple endpoints, add delays to avoid overwhelming the API server.

**Key Concepts:**
- **Rate Limiting**: Control request frequency
- **Delay Between Requests**: Use `time.sleep()` to pause
- **Respectful Usage**: Free APIs are a gift - don't abuse them!


In [ ]:
import time

def fetch_with_retry(url, max_retries=3, timeout=10):
    """
    Fetch data with exponential backoff retry logic.
    
    Parameters:
    -----------
    url : str
        URL to fetch
    max_retries : int
        Maximum number of retry attempts
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None if all retries failed
    """
    for attempt in range(max_retries):
        try:
            print(f"🔄 Attempt {attempt + 1}/{max_retries}...")
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            print("✅ Success!")
            return response.json()
            
        except requests.exceptions.HTTPError as e:
            # Don't retry client errors (4xx)
            if 400 <= e.response.status_code < 500:
                print(f"❌ Client error {e.response.status_code}: {e}")
                return None
            # Retry server errors (5xx)
            print(f"⚠️ Server error {e.response.status_code}: {e}")
            
        except requests.exceptions.Timeout:
            print(f"⚠️ Timeout after {timeout} seconds")
            
        except requests.exceptions.ConnectionError as e:
            print(f"⚠️ Connection error: {e}")
            
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Unexpected error: {e}")
        
        # Exponential backoff: wait 2^attempt seconds
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"⏳ Waiting {wait_time} seconds before retry...\n")
            time.sleep(wait_time)
    
    print("❌ All retries failed")
    return None

# Test it!
print("Testing retry logic with bad URL:")
test_url = "http://api.citybik.es/v2/networks/fake-network-12345"
result = fetch_with_retry(test_url, max_retries=3)

---

## Part 5a: Retry Logic & Rate Limiting

When working with external APIs in production, you need to handle two common challenges:

1. **Temporary Failures**: APIs can be temporarily unavailable or slow
2. **Rate Limits**: APIs restrict how many requests you can make

Let's implement production-ready patterns for both.

---

### Pattern 1: Retry Logic with Exponential Backoff

**Key Concepts:**
- **Exponential Backoff**: Wait progressively longer between retries (1s, 2s, 4s, 8s...)
- **Selective Retry**: Only retry on errors that might resolve (5xx server errors, timeouts)
- **Max Retries**: Limit attempts to avoid infinite loops

**When to Retry:**
- ✅ 5xx Server Errors (temporary server issues)
- ✅ Network timeouts
- ✅ Connection errors
- ❌ 4xx Client Errors (your request is wrong - retry won't help!)


### 🎯 Best Practices Summary

**Retry Logic:**
- ✅ Retry on temporary failures (timeouts, 5xx errors)
- ❌ Don't retry on client errors (4xx - it won't help!)
- ✅ Use exponential backoff (1s, 2s, 4s, 8s...)
- ✅ Limit max retries (3-5 is reasonable)
- ✅ Log each attempt for debugging

**Rate Limiting:**
- ✅ Add delays between bulk requests (1-2 seconds)
- ✅ Check API documentation for limits
- ✅ Consider using `time.sleep()` or `asyncio` for delays
- ✅ Be respectful of free APIs - they're a gift!

**Production Tips:**
- Use `requests.Session()` for connection pooling
- Consider `backoff` library for complex retry logic
- Implement circuit breakers for persistent failures
- Monitor and log API usage patterns

---

### 💡 When to Use These Patterns

**Use Retry Logic When:**
- Calling external APIs (especially free/public ones)
- Network conditions are unreliable
- Building production data pipelines
- You need resilient data collection

**Use Rate Limiting When:**
- Fetching from multiple endpoints
- API has documented rate limits
- You're making many requests in a loop
- Building automated/scheduled jobs

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5B. RATE LIMITING FOR BULK REQUESTS
# ═══════════════════════════════════════════════════════════

def fetch_multiple_networks_safe(network_ids, delay=1.0):
    """
    Fetch data for multiple networks with rate limiting.
    
    Parameters:
    -----------
    network_ids : list
        List of network IDs to fetch
    delay : float
        Seconds to wait between requests
    
    Returns:
    --------
    dict
        Network ID -> data mapping
    """
    import time
    
    results = {}
    
    for i, network_id in enumerate(network_ids):
        url = f"{BASE_URL}/networks/{network_id}"
        
        print(f"\n📡 Fetching {network_id} ({i+1}/{len(network_ids)})...")
        
        data = fetch_with_retry(url, max_retries=2)
        
        if data:
            results[network_id] = data
            print(f"✅ Fetched {len(data['network']['stations'])} stations")
        else:
            print(f"❌ Failed to fetch {network_id}")
        
        # Rate limiting: wait before next request (except after last one)
        if i < len(network_ids) - 1:
            print(f"⏳ Rate limit delay: {delay}s...")
            time.sleep(delay)
    
    return results


# Example: Fetch data for multiple Dutch bike networks
dutch_networks = ['ns-bikes', 'ov-fiets']

print("Fetching multiple networks with rate limiting:")
print("=" * 60)
network_data = fetch_multiple_networks_safe(dutch_networks, delay=1.0)

print(f"\n📊 Summary:")
print(f"   Total networks fetched: {len(network_data)}")
for network_id, data in network_data.items():
    print(f"   • {network_id}: {len(data['network']['stations'])} stations")

### 🧠 Pattern 2: Rate Limiting for Bulk Requests

When fetching data from multiple endpoints, add delays to avoid rate limits.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5A. RETRY LOGIC WITH EXPONENTIAL BACKOFF
# ═══════════════════════════════════════════════════════════

def fetch_with_retry(url, max_retries=3, timeout=10):
    """
    Fetch data with retry logic and exponential backoff.
    
    Parameters:
    -----------
    url : str
        URL to fetch
    max_retries : int
        Maximum number of retry attempts
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    import time
    
    for attempt in range(max_retries):
        try:
            print(f"🔄 Attempt {attempt + 1}/{max_retries}...")
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()
            
            print(f"✅ Success on attempt {attempt + 1}!")
            return response.json()
        
        except requests.exceptions.Timeout:
            print(f"⏱️ Timeout on attempt {attempt + 1}")
            
        except requests.exceptions.HTTPError as e:
            # Don't retry on 4xx errors (client errors)
            if 400 <= e.response.status_code < 500:
                print(f"❌ Client error {e.response.status_code} - not retrying")
                return None
            print(f"⚠️ Server error on attempt {attempt + 1}: {e}")
            
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request failed on attempt {attempt + 1}: {e}")
        
        # Exponential backoff: wait 2^attempt seconds before retry
        if attempt < max_retries - 1:  # Don't sleep after last attempt
            wait_time = 2 ** attempt
            print(f"⏳ Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)
    
    print("❌ All retry attempts failed")
    return None


# Test the retry logic
print("Testing retry logic with valid endpoint:")
print("=" * 60)
data = fetch_with_retry(NETWORK_URL, max_retries=3)

if data:
    print(f"\n✅ Successfully fetched data for {data['network']['name']}")
else:
    print("\n❌ Failed to fetch data after all retries")

---

## 🔄 Part 5a: Retry Logic & Rate Limiting (Production Patterns)

### Why Retry Logic Matters

**Real-World API Challenges:**
- Network hiccups cause temporary failures
- APIs occasionally timeout or return 5xx errors
- Rate limits require waiting between requests

**Solution**: Implement smart retry logic with exponential backoff

---

### 📚 Key Concepts

**Exponential Backoff:**
- First retry: Wait 1 second
- Second retry: Wait 2 seconds  
- Third retry: Wait 4 seconds
- Prevents hammering failing servers

**Rate Limiting:**
- Add delays between requests
- Respect API quotas
- Be a good API citizen

---

### 🧠 Pattern 1: Simple Retry with Exponential Backoff

In [ ]:
# ═══════════════════════════════════════════════════════════
# SOLUTION: ERROR HANDLING
# ═══════════════════════════════════════════════════════════

def fetch_bike_data_safe_SOLUTION(network_id, timeout=10):
    """
    Fetch bike data with error handling - SOLUTION VERSION.
    """
    url = f"{BASE_URL}/networks/{network_id}"
    
    try:
        # Make request with timeout
        response = requests.get(url, timeout=timeout)
        
        # Raise exception for bad HTTP status
        response.raise_for_status()
        
        # Parse and return JSON
        return response.json()
        
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.ConnectionError:
        print("🔌 Connection Error: Could not connect to API")
        return None
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error: {e}")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Request Error: {e}")
        return None
        
    except json.JSONDecodeError:
        print("📝 JSON Error: Could not parse response")
        return None


# Use solution version for rest of notebook
fetch_bike_data_safe = fetch_bike_data_safe_SOLUTION

# Fetch fresh data
print("📡 Fetching data with error handling...")
data = fetch_bike_data_safe(NETWORK_ID)

if data:
    print(f"✅ Successfully fetched {len(data['network']['stations'])} stations")
else:
    print("❌ Failed to fetch data")

---

## 📊 Part 5: Convert to DataFrame

### 🧠 Your Task: Convert JSON to DataFrame

**What You've Learned in Module 1:**
- In [M1_R1_fetching_bike_data.ipynb](../Module_01_Introduction/M1_R1_fetching_bike_data.ipynb) and [M1_03_open_data_sources.ipynb](../Module_01_Introduction/M1_03_open_data_sources.ipynb), you saw:
  - How to extract data from nested JSON (like `data['network']['stations']`)
  - How to create DataFrames from lists of dictionaries: `pd.DataFrame(list_of_dicts)`
  - How to inspect DataFrames with `.head()`, `.info()`, `.shape`

**Now It's Your Turn:**
Complete the code below to convert the station data to a DataFrame.

**Steps:**
1. Extract the stations list from `data['network']['stations']`
2. Convert to a DataFrame using `pd.DataFrame()`
3. Display the first few rows and basic info

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. CONVERT TO DATAFRAME - YOUR TASK
# ═══════════════════════════════════════════════════════════

# TODO: Extract stations list from the nested JSON
# Hint: You saw this pattern in M1_R1 - data['network']['stations']
stations = None  # Replace with your code

# TODO: Convert to DataFrame
# Hint: pd.DataFrame(list_of_dictionaries)
df_bikes = None  # Replace with your code

# TODO: Display information
# Hint: Use .head() to see first rows
# Hint: Use .info() to see column types and null counts
# Hint: Use len() to count rows

# Your code here...

In [ ]:
# ═══════════════════════════════════════════════════════════
# SOLUTION: CONVERT TO DATAFRAME
# ═══════════════════════════════════════════════════════════

# Extract stations list
stations = data['network']['stations']

# Convert to DataFrame
df_bikes = pd.DataFrame(stations)

print(f"✅ Created DataFrame with {len(df_bikes)} rows and {len(df_bikes.columns)} columns")
print(f"\n📊 First few rows:")
display(df_bikes.head())

print("\n" + "=" * 60)
print("📋 Dataset Info:")
print("=" * 60)
df_bikes.info()

### ✅ Solution: Convert to DataFrame

Run this cell after attempting the task above.

### Data Cleaning and Transformation

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. DATA CLEANING
# ═══════════════════════════════════════════════════════════

# Parse timestamp to datetime
df_bikes['timestamp'] = pd.to_datetime(df_bikes['timestamp'])

# Rename columns for clarity
df_bikes = df_bikes.rename(columns={
    'free_bikes': 'bikes_available',
    'empty_slots': 'docks_available'
})

# Add derived columns
df_bikes['total_capacity'] = df_bikes['bikes_available'] + df_bikes['docks_available']
df_bikes['utilization_pct'] = (df_bikes['bikes_available'] / df_bikes['total_capacity'] * 100).round(2)

# Add metadata
df_bikes['network_id'] = data['network']['id']
df_bikes['network_name'] = data['network']['name']
df_bikes['city'] = data['network']['location']['city']
df_bikes['country'] = data['network']['location']['country']

# Reorder columns
column_order = [
    'id', 'name', 'latitude', 'longitude',
    'bikes_available', 'docks_available', 'total_capacity', 'utilization_pct',
    'timestamp', 'network_id', 'network_name', 'city', 'country'
]

# Keep only columns that exist (some might be missing)
column_order = [col for col in column_order if col in df_bikes.columns]
df_bikes = df_bikes[column_order]

print("✅ Data cleaned and transformed")
print(f"\n📊 Final Dataset Preview:")
display(df_bikes.head(10))

---

## 🔍 Part 6: Data Exploration

Let's explore the bike availability data!

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. SUMMARY STATISTICS
# ═══════════════════════════════════════════════════════════

print("📈 Summary Statistics:")
print("=" * 60)
display(df_bikes[['bikes_available', 'docks_available', 'total_capacity', 'utilization_pct']].describe())

print("\n" + "=" * 60)
print("🚴 Bike Availability Summary:")
print("=" * 60)
print(f"Total Stations: {len(df_bikes)}")
print(f"Total Bikes Available: {df_bikes['bikes_available'].sum()}")
print(f"Total Docks Available: {df_bikes['docks_available'].sum()}")
print(f"Average Station Capacity: {df_bikes['total_capacity'].mean():.1f}")
print(f"Average Utilization: {df_bikes['utilization_pct'].mean():.1f}%")

print("\n" + "=" * 60)
print("⚠️ Stations with Issues:")
print("=" * 60)
empty_stations = df_bikes[df_bikes['bikes_available'] == 0]
full_stations = df_bikes[df_bikes['docks_available'] == 0]
print(f"Empty stations (no bikes): {len(empty_stations)}")
print(f"Full stations (no docks): {len(full_stations)}")

### Visualizations

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. VISUALIZATIONS
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('🚴 Amsterdam Bike Sharing - Real-time Overview', fontsize=16, fontweight='bold')

# 1. Distribution of bikes available
axes[0, 0].hist(df_bikes['bikes_available'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Bikes Available')
axes[0, 0].set_ylabel('Number of Stations')
axes[0, 0].set_title('Distribution of Bike Availability')
axes[0, 0].axvline(df_bikes['bikes_available'].mean(), color='red', linestyle='--', 
                   label=f"Mean: {df_bikes['bikes_available'].mean():.1f}")
axes[0, 0].legend()

# 2. Distribution of utilization
axes[0, 1].hist(df_bikes['utilization_pct'], bins=20, color='darkorange', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Utilization (%)')
axes[0, 1].set_ylabel('Number of Stations')
axes[0, 1].set_title('Station Utilization Distribution')
axes[0, 1].axvline(df_bikes['utilization_pct'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df_bikes['utilization_pct'].mean():.1f}%")
axes[0, 1].legend()

# 3. Top 10 stations by bikes available
top10 = df_bikes.nlargest(10, 'bikes_available')[['name', 'bikes_available']]
axes[1, 0].barh(range(len(top10)), top10['bikes_available'], color='green', alpha=0.7)
axes[1, 0].set_yticks(range(len(top10)))
axes[1, 0].set_yticklabels(top10['name'], fontsize=8)
axes[1, 0].set_xlabel('Bikes Available')
axes[1, 0].set_title('Top 10 Stations by Bike Availability')
axes[1, 0].invert_yaxis()

# 4. Capacity analysis
capacity_data = df_bikes[['bikes_available', 'docks_available']].mean()
axes[1, 1].bar(['Bikes\nAvailable', 'Docks\nAvailable'], capacity_data, 
               color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('Average Count')
axes[1, 1].set_title('Average Bikes vs Docks Available')
for i, v in enumerate(capacity_data):
    axes[1, 1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizations created")

### 🧠 Learner Task 2: Create a Station Map

**Your Task**: Create a scatter plot showing station locations on a map.

Requirements:
1. Use `latitude` and `longitude` columns
2. Color points by `bikes_available`
3. Size points by `total_capacity`
4. Add labels and legend

**Hint**: Use `plt.scatter()` with `c=` for color and `s=` for size parameters.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. LEARNER TASK 2: STATION MAP
# ═══════════════════════════════════════════════════════════

# TODO: Create a map of stations
# Hint: plt.scatter(x=longitude, y=latitude, c=bikes_available, s=total_capacity*5)

plt.figure(figsize=(12, 10))

# TODO: Your code here
# Example:
# plt.scatter(df_bikes['longitude'], df_bikes['latitude'], ...)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('🗺️ Amsterdam Bike Stations Map')
plt.colorbar(label='Bikes Available')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ✅ Solution: Station Map

In [ ]:
# ═══════════════════════════════════════════════════════════
# SOLUTION: STATION MAP
# ═══════════════════════════════════════════════════════════

plt.figure(figsize=(12, 10))

scatter = plt.scatter(
    df_bikes['longitude'], 
    df_bikes['latitude'],
    c=df_bikes['bikes_available'],
    s=df_bikes['total_capacity'] * 5,
    alpha=0.6,
    cmap='RdYlGn',
    edgecolor='black',
    linewidth=0.5
)

plt.colorbar(scatter, label='Bikes Available')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('🗺️ Amsterdam Bike Stations Map\n(Size = Capacity, Color = Availability)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add annotation for largest station
largest_station = df_bikes.nlargest(1, 'total_capacity').iloc[0]
plt.annotate(
    largest_station['name'],
    xy=(largest_station['longitude'], largest_station['latitude']),
    xytext=(10, 10), textcoords='offset points',
    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0')
)

plt.tight_layout()
plt.show()

print("✅ Station map created")

---

## ✅ Part 7: Data Validation

Let's validate our data quality before saving.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. DATA VALIDATION
# ═══════════════════════════════════════════════════════════

def validate_bike_data(df):
    """
    Validate bike availability data quality.
    
    Returns:
    --------
    bool : True if all validations pass
    """
    validations = []
    
    # Check 1: Required columns exist
    required_cols = ['id', 'name', 'bikes_available', 'timestamp']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
        validations.append(False)
    else:
        print(f"✅ All required columns present")
        validations.append(True)
    
    # Check 2: No negative values
    if (df['bikes_available'] < 0).any():
        print(f"❌ Found negative bike counts")
        validations.append(False)
    else:
        print(f"✅ No negative bike counts")
        validations.append(True)
    
    # Check 3: Timestamps are valid
    if df['timestamp'].isna().any():
        print(f"❌ Found missing timestamps")
        validations.append(False)
    else:
        print(f"✅ All timestamps present")
        validations.append(True)
    
    # Check 4: Coordinates are valid
    valid_lat = df['latitude'].between(-90, 90).all()
    valid_lon = df['longitude'].between(-180, 180).all()
    if not (valid_lat and valid_lon):
        print(f"❌ Invalid coordinates found")
        validations.append(False)
    else:
        print(f"✅ All coordinates valid")
        validations.append(True)
    
    # Check 5: No duplicates
    if df.duplicated(subset=['id']).any():
        print(f"❌ Found duplicate station IDs")
        validations.append(False)
    else:
        print(f"✅ No duplicate stations")
        validations.append(True)
    
    # Check 6: Reasonable data ranges
    if df['bikes_available'].max() > 100:
        print(f"⚠️ Warning: Unusually high bike count ({df['bikes_available'].max()})")
    
    return all(validations)


# Run validation
print("🔍 Validating data quality...")
print("=" * 60)
is_valid = validate_bike_data(df_bikes)
print("=" * 60)

if is_valid:
    print("✅ All validation checks passed!")
else:
    print("❌ Some validation checks failed - review data before saving")

---

## 💾 Part 8: Save Raw Data

Save both the raw JSON response and the cleaned DataFrame.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. SAVE RAW DATA
# ═══════════════════════════════════════════════════════════

# Save raw JSON response
print("💾 Saving raw JSON data...")
with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=2)
print(f"✅ Saved raw JSON: {OUTPUT_FILE}")
print(f"📦 File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")

# Also save as CSV for easy viewing
csv_file = OUTPUT_FILE.with_suffix('.csv')
df_bikes.to_csv(csv_file, index=False)
print(f"✅ Saved CSV: {csv_file}")
print(f"📦 File size: {csv_file.stat().st_size / 1024:.1f} KB")

# Create metadata file
metadata = {
    'source': 'CityBikes API',
    'api_url': NETWORK_URL,
    'network_id': NETWORK_ID,
    'network_name': data['network']['name'],
    'fetch_timestamp': datetime.now().isoformat(),
    'num_stations': len(df_bikes),
    'total_bikes_available': int(df_bikes['bikes_available'].sum()),
    'data_timestamp': df_bikes['timestamp'].iloc[0].isoformat(),
    'city': data['network']['location']['city'],
    'country': data['network']['location']['country']
}

metadata_file = OUTPUT_FILE.with_suffix('.metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved metadata: {metadata_file}")

print("\n" + "=" * 60)
print("📊 Data Acquisition Summary:")
print("=" * 60)
print(f"Stations fetched: {len(df_bikes)}")
print(f"Total bikes available: {df_bikes['bikes_available'].sum()}")
print(f"Files saved:")
print(f"  • {OUTPUT_FILE.name}")
print(f"  • {csv_file.name}")
print(f"  • {metadata_file.name}")

---

## 📝 Part 9: Summary

### What You've Learned ✅

In this notebook, you:
1. ✅ Connected to the CityBikes REST API
2. ✅ Implemented error handling for API requests
3. ✅ Parsed JSON responses and converted to DataFrames
4. ✅ Performed data cleaning and transformation
5. ✅ Validated data quality
6. ✅ Visualized bike availability patterns
7. ✅ Saved raw data with proper documentation

### Key Takeaways 💡

1. **APIs are just URLs** that return structured data (usually JSON)
2. **Error handling is essential** - networks fail, APIs change, timeouts happen
3. **Always validate data** before saving or analyzing
4. **Document your data sources** - future you will thank current you!
5. **Save raw data** before any transformations (immutable source of truth)

### Data Files Created 📁

- `data/raw/amsterdam_bike_{timestamp}.json` - Raw API response
- `data/raw/amsterdam_bike_{timestamp}.csv` - Cleaned DataFrame
- `data/raw/amsterdam_bike_{timestamp}.metadata.json` - Data documentation

### Next Steps 🚀

Now that you have bike data, proceed to:
- **M2_02_weather_data_api.ipynb** - Fetch weather data
- **M2_03_data_storage.ipynb** - Learn storage best practices
- **M2_04_merge_datasets.ipynb** - Combine bike and weather data

### 🧠 Reflection Questions

1. **What challenges might arise** from fetching real-time data at different times of day?
2. **How would you handle API rate limits** if you needed to fetch data every minute?
3. **What other validation checks** could you add to ensure data quality?
4. **How could you automate** this data collection process?

**Write your reflections below** ⬇️

### My Reflections

[Your thoughts here]

---

## 📚 References

- [CityBikes API Documentation](http://api.citybik.es/v2/)
- [Requests Library Documentation](https://requests.readthedocs.io/)
- [Pandas I/O Tools](https://pandas.pydata.org/docs/user_guide/io.html)
- [REST API Tutorial](https://restfulapi.net/)
- [JSON Format Guide](https://www.json.org/json-en.html)

---

**🎉 Congratulations!** You've successfully completed M2_01 - Amsterdam Bike Data Acquisition!